#Step 1: Creating the Governance Schema

In [0]:
%sql
-- 1. Create the Schema
CREATE SCHEMA IF NOT EXISTS maven_catalog.governance;

-- ==========================================================
-- 1. DATA ENGINEERS (Safe RW Access - No DROP SCHEMA)
-- ==========================================================
GRANT USE CATALOG ON CATALOG maven_catalog TO `grp_data_engineers`;

-- Grant Entry to all schemas
GRANT USE SCHEMA ON SCHEMA maven_catalog.bronze_schema TO `grp_data_engineers`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.silver_schema TO `grp_data_engineers`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.gold_schema   TO `grp_data_engineers`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.governance    TO `grp_data_engineers`;

-- Grant Safe "Read-Write" (Using 'CREATE' to cover Views/Functions for v1.0)
GRANT SELECT, MODIFY, CREATE TABLE, CREATE 
ON SCHEMA maven_catalog.bronze_schema TO `grp_data_engineers`;

GRANT SELECT, MODIFY, CREATE TABLE, CREATE 
ON SCHEMA maven_catalog.silver_schema TO `grp_data_engineers`;

GRANT SELECT, MODIFY, CREATE TABLE, CREATE 
ON SCHEMA maven_catalog.gold_schema TO `grp_data_engineers`;

GRANT SELECT, MODIFY, CREATE TABLE, CREATE 
ON SCHEMA maven_catalog.governance TO `grp_data_engineers`;

-- ==========================================================
-- 2. ANALYSTS & EXECUTIVES (Read-Only Gold)
-- ==========================================================
-- Analyst Group
GRANT USE CATALOG ON CATALOG maven_catalog TO `grp_analysts`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.gold_schema TO `grp_analysts`;
GRANT SELECT ON SCHEMA maven_catalog.gold_schema TO `grp_analysts`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.governance TO `grp_analysts`;
GRANT EXECUTE ON SCHEMA maven_catalog.governance TO `grp_analysts`;

-- Executive Group
GRANT USE CATALOG ON CATALOG maven_catalog TO `grp_executives`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.gold_schema TO `grp_executives`;
GRANT SELECT ON SCHEMA maven_catalog.gold_schema TO `grp_executives`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.governance TO `grp_executives`;
GRANT EXECUTE ON SCHEMA maven_catalog.governance TO `grp_executives`;

-- ==========================================================
-- 3. REGIONAL GROUPS (Read-Only Gold + RLS)
-- ==========================================================

-- Region 1: Mexico West
GRANT USE CATALOG ON CATALOG maven_catalog TO `Mexico West`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.gold_schema TO `Mexico West`;
GRANT SELECT ON SCHEMA maven_catalog.gold_schema TO `Mexico West`;
GRANT EXECUTE ON SCHEMA maven_catalog.governance TO `Mexico West`;

-- Region 2: North West
GRANT USE CATALOG ON CATALOG maven_catalog TO `North West`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.gold_schema TO `North West`;
GRANT SELECT ON SCHEMA maven_catalog.gold_schema TO `North West`;
GRANT EXECUTE ON SCHEMA maven_catalog.governance TO `North West`;

-- Region 3: Mexico Central
GRANT USE CATALOG ON CATALOG maven_catalog TO `Mexico Central`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.gold_schema TO `Mexico Central`;
GRANT SELECT ON SCHEMA maven_catalog.gold_schema TO `Mexico Central`;
GRANT EXECUTE ON SCHEMA maven_catalog.governance TO `Mexico Central`;

-- Region 4: South West
GRANT USE CATALOG ON CATALOG maven_catalog TO `South West`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.gold_schema TO `South West`;
GRANT SELECT ON SCHEMA maven_catalog.gold_schema TO `South West`;
GRANT EXECUTE ON SCHEMA maven_catalog.governance TO `South West`;

-- Region 5: Mexico South
GRANT USE CATALOG ON CATALOG maven_catalog TO `Mexico South`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.gold_schema TO `Mexico South`;
GRANT SELECT ON SCHEMA maven_catalog.gold_schema TO `Mexico South`;
GRANT EXECUTE ON SCHEMA maven_catalog.governance TO `Mexico South`;

-- Region 6: Central West
GRANT USE CATALOG ON CATALOG maven_catalog TO `Central West`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.gold_schema TO `Central West`;
GRANT SELECT ON SCHEMA maven_catalog.gold_schema TO `Central West`;
GRANT EXECUTE ON SCHEMA maven_catalog.governance TO `Central West`;

-- Region 7: Canada West
GRANT USE CATALOG ON CATALOG maven_catalog TO `Canada West`;
GRANT USE SCHEMA ON SCHEMA maven_catalog.gold_schema TO `Canada West`;
GRANT SELECT ON SCHEMA maven_catalog.gold_schema TO `Canada West`;
GRANT EXECUTE ON SCHEMA maven_catalog.governance TO `Canada West`;


#CREATING A RLS POLICY

In [0]:
%sql
CREATE OR REPLACE FUNCTION maven_catalog.governance.rls_region_filter(region_col STRING)
RETURNS BOOLEAN
RETURN 
  -- CRITICAL FIX: Change 'data engineers' to 'grp_data_engineers'
  is_account_group_member('grp_data_engineers') OR 
  is_account_group_member('grp_executives') OR
  is_account_group_member('grp_analysts') OR 
  
  -- Dynamic check for Regional Managers
  is_account_group_member(region_col);

In [0]:
%sql
DROP FUNCTION IF EXISTS maven_catalog.governance.cls_mask_pii;

In [0]:
%sql
ALTER MATERIALIZED VIEW maven_catalog.gold_schema.fact_sales 
SET ROW FILTER maven_catalog.governance.rls_region_filter ON (sales_region);

#CREATING A CLS POLICY

In [0]:
%sql
CREATE OR REPLACE FUNCTION maven_catalog.governance.cls_mask_pii(col_value STRING)
RETURNS STRING
RETURN CASE 
  -- Whitelist: Who sees real data?
  WHEN is_account_group_member('grp_data_engineers') THEN col_value
  WHEN is_account_group_member('grp_executives') THEN col_value
  
  -- Blacklist: Everyone else (Analysts + All 7 Regional Groups)
  ELSE '****'
END;

In [0]:
%sql
-- Apply to customer_acct_num Column
ALTER MATERIALIZED VIEW maven_catalog.gold_schema.dim_customers
ALTER COLUMN customer_acct_num SET MASK maven_catalog.governance.cls_mask_pii;

-- -- Apply to Credit Card Column (Ensure the column name matches your table)
-- ALTER TABLE maven_catalog.gold_schema.dim_customers 
-- ALTER COLUMN credit_card SET MASK maven_catalog.governance.cls_mask_pii;

In [0]:
%sql
use catalog maven_catalog;
use schema gold_schema;

In [0]:
%sql
select current_user();


In [0]:
%sql
select *
from dim_stores limit 20


In [0]:
%sql
describe catalog maven_catalog;

In [0]:
%sql
DROP SCHEMA IF EXISTS maven_catalog.governance;